In [0]:
%sql
MERGE INTO la_lakehouse.gold.fact_permits AS target 
USING (
    SELECT 
    s.permit_nbr,
    s.valuation,
    s.square_footage,
    s.du_changed,
    s.zip_code,
    s.adu_changed,
    s.height,
    s.primary_address,
    s.work_desc,
    s.status_desc,
    s.status_date,
    s.construction,
    s.lat,
    s.lon,
    s.type_lat_lon,
    s.geolocation,
    s.pin_nbr,
    s.square_footage_multi,
    s.zone_is_multi,
    s.ct_multi,
    issue_date.date_key AS issue_date_key,
    cofo_date.date_key AS cofo_date_key,
    submitted_date.date_key AS submitted_date_key,
    dz.zone_key,
    dct.ct_key,
    dd.district_key,
    dapc.apc_key,
    dcpa.cpa_key,
    dcnc.cnc_key,
    dhoa.hl_key,
    dpt.permit_type_key,
    dut.use_type_key
    FROM la_lakehouse.silver.silver_permits_issued_enriched AS s
    LEFT JOIN la_lakehouse.gold.dim_zone AS dz
    ON s.zone_base = dz.zone_base
    LEFT JOIN la_lakehouse.gold.dim_census_tract AS dct
    ON s.ct = dct.ct
    LEFT JOIN la_lakehouse.gold.dim_district AS dd
    ON s.cd = dd.cd
    LEFT JOIN la_lakehouse.gold.dim_area_planning_commision AS dapc
    ON s.apc = dapc.apc
    LEFT JOIN la_lakehouse.gold.dim_community_plan_area AS dcpa
    ON s.cpa = dcpa.cpa
    LEFT JOIN la_lakehouse.gold.dim_certified_neighborhood_council AS dcnc
    ON s.cnc = dcnc.cnc
    LEFT JOIN la_lakehouse.gold.dim_hillside_ordinance_area AS dhoa
    ON s.hl = dhoa.hl
    LEFT JOIN la_lakehouse.gold.dim_permit_type AS dpt
    ON s.permit_group = dpt.permit_group 
    AND s.permit_type = dpt.permit_type
    AND s.permit_sub_type = dpt.permit_sub_type 
    LEFT JOIN la_lakehouse.gold.dim_use_type AS dut 
    ON s.use_code = dut.use_code AND s.use_desc = dut.use_desc
    LEFT JOIN la_lakehouse.gold.dim_date AS submitted_date
    ON DATE(s.submitted_date) = submitted_date.full_date
    LEFT JOIN la_lakehouse.gold.dim_date AS issue_date
    ON DATE(s.issue_date) = issue_date.full_date
    LEFT JOIN la_lakehouse.gold.dim_date AS cofo_date
    ON DATE(s.cofo_date) = cofo_date.full_date
) AS source
ON target.permit_nbr = source.permit_nbr
WHEN NOT MATCHED THEN 
    INSERT(
    permit_nbr,
    valuation,
    square_footage,
    du_changed,
    zip_code,
    adu_changed,
    height,
    primary_address,
    work_desc,
    status_desc,
    status_date,
    construction,
    lat,
    lon,
    type_lat_lon,
    geolocation,
    pin_nbr,
    square_footage_multi,
    zone_is_multi,
    ct_multi,
    submitted_date_key,
    issue_date_key,
    cofo_date_key,
    zone_key,
    ct_key,
    district_key,
    apc_key,
    cpa_key,
    cnc_key,
    hl_key,
    permit_type_key,
    use_type_key
    )
    VALUES(
    source.permit_nbr,
    source.valuation,
    source.square_footage,
    source.du_changed,
    source.zip_code,
    source.adu_changed,
    source.height,
    source.primary_address,
    source.work_desc,
    source.status_desc,
    source.status_date,
    source.construction,
    source.lat,
    source.lon,
    source.type_lat_lon,
    source.geolocation,
    source.pin_nbr,
    source.square_footage_multi,
    source.zone_is_multi,
    source.ct_multi,
    source.submitted_date_key,
    source.issue_date_key,
    source.cofo_date_key,
    source.zone_key,
    source.ct_key,
    source.district_key,
    source.apc_key,
    source.cpa_key,
    source.cnc_key,
    source.hl_key,
    source.permit_type_key,
    source.use_type_key
    );

#Testing

In [0]:
%sql
SELECT COUNT(*) FROM la_lakehouse.gold.fact_permits

In [0]:
%sql
SELECT 
COUNT(district_key) AS num_districts,
COUNT(submitted_date_key) AS num_submitted,
COUNT(issue_date_key) AS num_issue,
COUNT(cofo_date_key) AS num_cofo,
COUNT(zone_key) AS num_zone,
COUNT(ct_key) AS num_ct,
COUNT(apc_key) AS num_apc,
COUNT(cpa_key) AS num_cpa,
COUNT(cnc_key) AS num_cnc,
COUNT(hl_key) AS num_hl,
COUNT(permit_type_key) AS num_pt,
COUNT(use_type_key) AS num_use
FROM la_lakehouse.gold.fact_permits
WHERE submitted_date_key IS NOT NULL
AND issue_date_key IS NOT NULL
AND cofo_date_key IS NOT NULL
AND zone_key IS NOT NULL
AND ct_key IS NOT NULL
AND apc_key IS NOT NULL
AND cpa_key IS NOT NULL
AND cnc_key IS NOT NULL
AND hl_key IS NOT NULL
AND permit_type_key IS NOT NULL
AND use_type_key IS NOT NULL;

In [0]:
%sql
SELECT 
COUNT(*) AS total,
COUNT(district_key) AS has_district,
COUNT(submitted_date_key) AS has_submitted,
COUNT(cofo_date_key) AS has_cofo,
COUNT(zone_key) AS has_zone,
COUNT(use_type_key) AS has_use
FROM la_lakehouse.gold.fact_permits;